# 03 - Experience B : interference des artefacts

On teste l'hypothese H2 : l'insertion est plus difficile a distinguer sur les images generees, car ses
traces se noient dans les artefacts de generation.

Cette experience ne demande aucun detecteur entraine. On mesure directement, sur les images, comment
l'insertion deplace la distribution des residus, et on compare l'ampleur de ce deplacement entre le
naturel et le genere. Elle fonctionne donc aussi pour les algorithmes adaptatifs, que l'on ne sait pas
detecter a cette echelle.

L'outil central est la taille d'effet, le d de Cohen : plus elle est grande, plus l'insertion deplace
nettement la statistique. Une taille d'effet plus faible sur le genere signale l'interference.

## Configuration

In [ ]:
# ================= CONFIGURATION =================
SOURCES  = ['natural', 'sd', 'sdxl', 'adm']
ALGO     = 'uniward'      # 'lsb', 'uniward' ou 'hill'
PAYLOAD  = 0.4            # charge utile
N        = 300           # images par classe, selon ce que contient le corpus
SEED     = 42
# ================================================
print('Interference, algorithme', ALGO, 'a', PAYLOAD, 'bpp')

## Environnement et corpus

In [ ]:
import os, glob, shutil
import numpy as np
np.random.seed(SEED)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/memoire_data'
    ROOT = '/content/corpus'
except Exception:
    DATA_DIR = os.path.abspath('./memoire_data')
    ROOT = os.path.abspath('./corpus')
RESULTS = f'{DATA_DIR}/results'
os.makedirs(RESULTS, exist_ok=True)

if not os.path.isdir(f'{ROOT}/natural/cover'):
    zips = sorted(glob.glob(f'{DATA_DIR}/corpus_*.zip'))
    if zips:
        shutil.unpack_archive(zips[-1], ROOT)
        print('Corpus restaure depuis', zips[-1])
print('Corpus :', ROOT)

In [ ]:
!pip install -q imageio scipy scikit-learn matplotlib
print('Installation terminee.')

## 1. Descripteur de residus

Pour chaque image, on calcule le residu haute frequence par un filtre passe haut, puis quatre statistiques
qui resument sa distribution : variance, asymetrie, kurtosis et entropie. Ce sont ces statistiques que
l'insertion perturbe, et c'est sur elles qu'on mesure l'interference.

In [ ]:
import imageio.v2 as imageio
from scipy.signal import convolve2d
from scipy.stats import skew, kurtosis

STATS = ['variance', 'asymetrie', 'kurtosis', 'entropie']

def residu_hf(img):
    # filtre passe haut simple, fait ressortir les traces d'insertion
    k = np.array([[-1, 2, -1], [2, -4, 2], [-1, 2, -1]], float)
    return convolve2d(img, k, mode='same', boundary='symm')

def descripteur(path):
    r = residu_hf(imageio.imread(path).astype(float)).ravel()
    counts, _ = np.histogram(r, bins=64)
    p = counts / counts.sum(); p = p[p > 0]
    entropie = float(-(p * np.log2(p)).sum())
    return [float(np.var(r)), float(skew(r)), float(kurtosis(r)), entropie]

def descripteurs_dossier(dossier, motif, n):
    fichiers = sorted(glob.glob(f'{dossier}/{motif}'))[:n]
    return np.array([descripteur(f) for f in fichiers])

print('Descripteur pret, dimensions :', STATS)

## 2. Calcul des descripteurs par classe

Pour chaque source, on calcule les descripteurs des images vierges et des images porteuses.

In [ ]:
desc_cover, desc_stego = {}, {}
for src in SOURCES:
    c = descripteurs_dossier(f'{ROOT}/{src}/cover', '*.pgm', N)
    s = descripteurs_dossier(f'{ROOT}/{src}/{ALGO}', f'*_p{PAYLOAD}.pgm', N)
    if len(c) == 0 or len(s) == 0:
        print(src, ': images manquantes, source ignoree'); continue
    m = min(len(c), len(s))
    desc_cover[src], desc_stego[src] = c[:m], s[:m]
    print(f'{src:8s}: {m} images par classe')

## 3. Tailles d'effet cover vers stego

Pour chaque source et chaque statistique, on mesure de combien l'insertion deplace la valeur, par le d de
Cohen, et on verifie que le deplacement est significatif par un test de Mann-Whitney. Un d plus faible sur
le genere que sur le naturel est la signature de l'interference.

In [ ]:
from scipy.stats import mannwhitneyu

def cohen_d(a, b):
    # ampleur de l'ecart entre deux groupes, en ecarts types
    s = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2)
    return (np.mean(b) - np.mean(a)) / s if s > 0 else 0.0

print(f"{'source':8s} " + ' '.join(f'{n:>10s}' for n in STATS) + '   |d| moyen')
resume = {}
for src in SOURCES:
    if src not in desc_cover:
        continue
    ds = []
    for j in range(len(STATS)):
        d = cohen_d(desc_cover[src][:, j], desc_stego[src][:, j])
        ds.append(abs(d))
    resume[src] = np.mean(ds)
    print(f'{src:8s} ' + ' '.join(f'{d:10.3f}' for d in ds) + f'   {resume[src]:.3f}')

print('\nComparaison de l\'ampleur moyenne du deplacement :')
if 'natural' in resume:
    for src in SOURCES:
        if src in resume and src != 'natural':
            rapport = resume[src] / resume['natural'] if resume['natural'] else float('nan')
            print(f'  {src:8s}: {resume[src]:.3f}  soit {rapport:.2f} fois le naturel')

## 4. Figures

Un violon du kurtosis pour voir le deplacement cover vers stego par source, et une projection en deux
dimensions pour voir le chevauchement des classes.

In [ ]:
import matplotlib.pyplot as plt

j_kurt = STATS.index('kurtosis')
fig, ax = plt.subplots(figsize=(11, 5))
positions, etiquettes, i = [], [], 0
for src in SOURCES:
    if src not in desc_cover:
        continue
    ax.violinplot([desc_cover[src][:, j_kurt], desc_stego[src][:, j_kurt]],
                  positions=[i, i + 1], widths=0.8, showmeans=True)
    etiquettes += [f'{src}\ncover', f'{src}\nstego']; positions += [i, i + 1]; i += 3
ax.set_xticks(positions); ax.set_xticklabels(etiquettes)
ax.set_ylabel('kurtosis du residu')
ax.set_title(f'Deplacement cover vers stego du kurtosis, {ALGO} {PAYLOAD} bpp')
plt.tight_layout(); plt.savefig(f'{RESULTS}/expB_violin_{ALGO}_p{PAYLOAD}.png', dpi=150); plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# On empile toutes les classes et on projette en deux dimensions
X, couleurs, noms = [], [], []
palette = {'natural': 'tab:blue', 'sd': 'tab:orange', 'sdxl': 'tab:green', 'adm': 'tab:red'}
for src in SOURCES:
    if src not in desc_cover:
        continue
    for classe, D in [('cover', desc_cover[src]), ('stego', desc_stego[src])]:
        X.append(D); couleurs += [palette[src]] * len(D)
        noms += [f'{src} {classe}'] * len(D)
X = np.vstack(X)
P = PCA(n_components=2, random_state=SEED).fit_transform(StandardScaler().fit_transform(X))

fig, ax = plt.subplots(figsize=(8, 7))
vus = set()
for i in range(0, len(P), max(1, len(P)//1500)):   # on allege l'affichage
    marqueur = 'o' if 'cover' in noms[i] else '^'
    lab = noms[i] if noms[i] not in vus else None; vus.add(noms[i])
    ax.scatter(P[i, 0], P[i, 1], c=couleurs[i], marker=marqueur, s=12, alpha=0.5, label=lab)
ax.set_title('Projection 2D des descripteurs, rond = cover, triangle = stego')
ax.legend(fontsize=8, markerscale=1.5); plt.tight_layout()
plt.savefig(f'{RESULTS}/expB_projection_{ALGO}_p{PAYLOAD}.png', dpi=150); plt.show()

## Interpretation

Ce qu'on attend : une taille d'effet moyenne plus faible sur les sources generees que sur le naturel,
signe que l'insertion y est plus masquee. Sur la projection, les paires cover et stego du naturel devraient
etre plus separees que celles des sources generees, dont les nuages se chevauchent davantage.

A relancer avec `ALGO = 'lsb'` et `'hill'`, et avec `PAYLOAD = 0.2`, pour comparer les algorithmes et les
charges. Noter les tailles d'effet dans le journal et copier les figures dans le dossier results du depot.